# Aula 4, RLHF

Notebook executável que acompanha a aula [04-rlhf.md](../../lessons/modulo-07-llms/04-rlhf.md).

## O que vamos fazer aqui

O coração do RLHF é o modelo de recompensa, que aprende a ranquear respostas a partir de
comparações humanas. Vamos construí-lo do zero: cada resposta tem características, um humano
simulado prefere a de maior qualidade, e o modelo aprende a prever essas preferências. Só
numpy.

## Modelo de recompensa a partir de preferências

A qualidade verdadeira de uma resposta é uma combinação das suas características, que o modelo
não conhece. A partir de muitos pares com preferências, treinamos um modelo logístico que
tenta descobrir os pesos.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# Qualidade verdadeira = combinação das características (o modelo não conhece estes pesos).
qualidade_real = np.array([1.0, 1.5, 0.5])   # clareza, correção, gentileza

n = 400
A = rng.uniform(0, 1, (n, 3))
B = rng.uniform(0, 1, (n, 3))
# Preferência humana: 1 se A é melhor que B segundo a qualidade real, com ruído.
pref = ((A @ qualidade_real + rng.normal(0, 0.1, n)) > (B @ qualidade_real)).astype(float)

# Modelo de recompensa: regressão logística sobre a diferença A - B.
w = np.zeros(3)
for _ in range(2000):
    dif = A - B
    p = 1 / (1 + np.exp(-(dif @ w)))
    w -= 0.5 * dif.T @ (p - pref) / n

dif = A - B
pred = (1 / (1 + np.exp(-(dif @ w))) > 0.5).astype(float)
print("Pesos verdadeiros        :", qualidade_real)
print("Pesos aprendidos          :", np.round(w, 2))
print("Acurácia nas preferências:", round(float((pred == pref).mean()), 3))

Os pesos aprendidos saem em escala maior que os verdadeiros (normal na regressão
logística), mas guardam as mesmas proporções: a característica de peso 1,5 continua a mais
importante. E o modelo prevê a preferência humana com acurácia em torno de 0,95. A partir
apenas de comparações do tipo "esta é melhor que aquela", ele destilou uma noção de
qualidade. É esse juiz que, no RLHF completo, guia o ajuste do LLM. Para o projeto, estude
como a acurácia cresce com o número de comparações.